# Can we predict 12-month loan default?
**Question:** using only information available at application time, can we predict `defaulted`?

Plan: data checks, baseline, logistic regression and gradient boosting with stratified CV, hold-out test evaluation, drivers, practical usefulness.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
SEED = 42
df = pd.read_csv('data.csv')
print(df.shape); df.head()

## 1. Data quality

In [ ]:
print('missing:', df.isna().sum().sum(), '| duplicate rows:', df.duplicated().sum(), '| duplicate ids:', df.application_id.duplicated().sum())
df.describe().T

In [ ]:
rate = df.defaulted.mean(); print(f'Default rate: {rate:.2%} ({df.defaulted.sum()} of {len(df)})')
print(df.groupby('region').defaulted.agg(['mean','count']).round(3))

No missing values or duplicates. The target is imbalanced (~10% positive), so accuracy is misleading; we use ROC-AUC and PR-AUC (a no-skill PR-AUC equals the base rate). `application_id` is an identifier and is excluded. There is no date column, so we use stratified random splits rather than a time-based split (a caveat for deployment on future cohorts).

## 2. Univariate relationships with default

In [ ]:
num = ['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']
fig, ax = plt.subplots(2, 3, figsize=(14, 7))
for a, c in zip(ax.ravel(), num):
    r = df.groupby(pd.qcut(df[c], 10, duplicates='drop'), observed=True).defaulted.mean()
    a.plot(range(len(r)), r.values, marker='o'); a.axhline(rate, ls='--', c='grey')
    a.set_title(c); a.set_xlabel('decile'); a.set_ylabel('default rate')
plt.tight_layout(); plt.show()

## 3. Models
20% hold-out test set; models compared with 5-fold stratified CV on the training portion only.

In [ ]:
features = ['region'] + num
X = df[features]; y = df.defaulted
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['region']), ('num', StandardScaler(), num)])
models = {
    'baseline (prior)': DummyClassifier(strategy='prior'),
    'logistic regression': Pipeline([('p', pre), ('m', LogisticRegression(max_iter=1000))]),
    'gradient boosting': Pipeline([('p', pre), ('m', HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, l2_regularization=1.0, random_state=SEED))]),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
rows = []
for n, m in models.items():
    p = cross_val_predict(m, Xtr, ytr, cv=cv, method='predict_proba')[:, 1]
    rows.append(dict(model=n, cv_roc_auc=roc_auc_score(ytr, p), cv_pr_auc=average_precision_score(ytr, p), cv_brier=brier_score_loss(ytr, p)))
pd.DataFrame(rows).set_index('model').round(4)

## 4. Final evaluation on held-out test set

In [ ]:
res = []; test_p = {}
for n, m in models.items():
    m.fit(Xtr, ytr); p = m.predict_proba(Xte)[:, 1]; test_p[n] = p
    res.append(dict(model=n, roc_auc=roc_auc_score(yte, p), pr_auc=average_precision_score(yte, p), brier=brier_score_loss(yte, p)))
display(pd.DataFrame(res).set_index('model').round(4))
rng = np.random.default_rng(SEED); p = test_p['logistic regression']; yv = yte.values
bs = [roc_auc_score(yv[i], p[i]) for i in (rng.integers(0, len(yv), len(yv)) for _ in range(500))]
lo, hi = np.percentile(bs, [2.5, 97.5]); print(f'Logistic test AUC 95% CI: {lo:.3f}-{hi:.3f}')
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for n, p in test_p.items():
    f, t, _ = roc_curve(yte, p); ax[0].plot(f, t, label=n)
    pr, rc, _ = precision_recall_curve(yte, p); ax[1].plot(rc, pr, label=n)
ax[0].plot([0, 1], [0, 1], 'k:'); ax[0].set(title='ROC', xlabel='FPR', ylabel='TPR')
ax[1].axhline(yte.mean(), c='k', ls=':'); ax[1].set(title='Precision-Recall', xlabel='recall', ylabel='precision')
ax[0].legend(); plt.show()

## 5. What drives the prediction?

In [ ]:
lr = models['logistic regression']
coef = pd.Series(lr.named_steps['m'].coef_[0], index=lr.named_steps['p'].get_feature_names_out()).sort_values()
print('Logistic coefficients (numerics standardised, log-odds):'); display(coef.round(3))
pi = permutation_importance(models['gradient boosting'], Xte, yte, scoring='roc_auc', n_repeats=20, random_state=SEED)
display(pd.Series(pi.importances_mean, index=features).sort_values(ascending=False).round(4).rename('GBM permutation importance (AUC drop)'))

## 6. Calibration and practical usefulness

In [ ]:
p = test_p['logistic regression']
d = pd.DataFrame({'p': p, 'y': yte.values}); d['bin'] = pd.qcut(d.p, 10, duplicates='drop')
display(d.groupby('bin', observed=True).agg(mean_pred=('p','mean'), obs_rate=('y','mean'), n=('y','size')).round(3))
order = d.sort_values('p', ascending=False); tot = d.y.sum()
for k in (0.1, 0.2, 0.3):
    top = order.head(int(len(d) * k)); print(f'Flag riskiest {k:.0%}: captures {top.y.sum()/tot:.0%} of defaults (lift {top.y.mean()/d.y.mean():.1f}x)')